# Next-Word Prediction on *A Study in Scarlet* (Arthur Conan Doyle)

**Goal:** compare four next-word predictors on the same book and the same held-out test set:

| # | Model | Type | Uses the book? |
|---|-------|------|----------------|
| 1 | Trigram + backoff | classic statistical baseline | trained on it |
| 2 | BERT (`bert-base-uncased`) | masked LM, zero-shot (`[MASK]` at the end) | no |
| 3 | GPT-2 (`gpt2`) | causal LM, zero-shot | no |
| 4 | GPT-2 **fine-tuned** | causal LM adapted to the book | fine-tuned on train split |

**Metrics:** top-1 / top-5 next-word accuracy (all models, same examples) and perplexity (GPT-2 before vs. after fine-tuning).

**Before you run:** `Runtime -> Change runtime type -> T4 GPU`, then run the cells top to bottom.

## 1. Setup

In [ ]:
!pip install -q transformers gradio
!apt-get -qq install -y poppler-utils > /dev/null   # provides `pdftotext` (usually already in Colab)

In [ ]:
import re, json, math, random, collections
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import (GPT2LMHeadModel, GPT2TokenizerFast, BertForMaskedLM, BertTokenizerFast,
                          get_linear_schedule_with_warmup)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "(switch to a GPU runtime for faster fine-tuning)" if device.type == "cpu" else "")

## 2. Load the book (PDF) and clean it

Upload `study_in_scarlet.pdf` when prompted. (Alternative: mount Google Drive and set `PDF_PATH` to the file there.)

Cleaning keeps **punctuation and capitalisation** (needed by BERT/GPT-2) and only removes PDF noise: the license/table-of-contents pages, page numbers, chapter headings and the broken drop-capital letters at chapter starts.

In [ ]:
from google.colab import files
uploaded = files.upload()                  # choose study_in_scarlet.pdf
PDF_PATH = list(uploaded.keys())[0]
# --- or, from Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# PDF_PATH = '/content/drive/MyDrive/study_in_scarlet.pdf'
print("Using:", PDF_PATH)

In [ ]:
import re, random, subprocess, collections

def extract_pdf_text(pdf_path):
    return subprocess.run(["pdftotext", pdf_path, "-"], capture_output=True, text=True, check=True).stdout

ABBREV = {"mr","mrs","ms","dr","st","messrs","m.d","no","vs","capt","col","gen","rev","hon","jr","sr","mt","j","h","a","b","c","d","e","f","g","k","l","n","o","p","r","s","t","u","w","x","y","z","i","v","m"}

def clean_book(raw):
    text = raw.replace("\f", "\n")
    m = re.search(r"PART I\.\n\(Being a reprint", text)
    if m: text = text[m.start():]
    text = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", text)                       # page numbers
    text = re.sub(r"(?m)^([A-Z])\n+(CHAPTER [IVX]+\.)\n(.+)\n+([a-z])", r"\2 \3\n\1\4", text)  # drop caps
    text = re.sub(r"PART I\.\n\(Being a reprint.*?Department\.\)", " ", text, flags=re.S)
    text = re.sub(r"PART II\.\nThe Country of the Saints\.", " ", text)
    text = re.sub(r"(?m)^PART II\.\s*$", " ", text)
    text = re.sub(r"(?m)^CHAPTER [IVX]+\.[ \t]+[^\n]+\n", " ", text)        # heading with title on same line (drop-cap chapters)
    text = re.sub(r"(?m)^CHAPTER [IVX]+\.\n[^\n]+\n", " ", text)           # heading + title on next line
    text = (text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'").replace("\u00a0", " "))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_sentences(text):
    parts = re.split(r'(?<=[.!?])(["\']?)\s+(?=["\']?[A-Z])', text)
    # re.split with a capture group returns [s, q, s, q, ...]; rebuild
    sents, cur = [], parts[0]
    for i in range(1, len(parts), 2):
        quote, nxt = parts[i], parts[i+1]
        cur_clean = cur
        last = re.findall(r"([A-Za-z.]+)[.!?]$", cur_clean)
        if cur_clean.endswith(".") and last and last[-1].lower().rstrip(".") in ABBREV and quote == "":
            cur = cur + quote + " " + nxt           # abbreviation (Mr., Dr., M.D., initials) -> not a sentence end
        else:
            sents.append(cur + quote); cur = nxt
    sents.append(cur)
    return [s.strip() for s in sents if len(s.split()) >= 3]

In [ ]:
raw = extract_pdf_text(PDF_PATH)
book = clean_book(raw)
sentences = split_sentences(book)

print(f"Words: {len(book.split()):,} | Sentences: {len(sentences):,}")
print("\nFirst sentences:")
for s in sentences[:3]: print(" -", s)
print("\nRandom sample:")
for s in random.sample(sentences, 5): print(" -", s)

# Save the cleaned corpus (nice artifact for GitHub)
with open("study_in_scarlet_clean.txt", "w", encoding="utf-8") as f: f.write(book)
with open("sentences.json", "w", encoding="utf-8") as f: json.dump(sentences, f, ensure_ascii=False, indent=1)

## 3. Train / validation / test split

The book is cut into chunks of 10 consecutive sentences, and **whole chunks** are assigned to 80% train / 10% validation / 10% test.
This keeps local context intact and avoids near-duplicate leakage between neighbouring sentences.
Validation is used to pick the best fine-tuning epoch; test is touched only for the final numbers.

In [ ]:
CHUNK = 10
chunks = [sentences[i:i + CHUNK] for i in range(0, len(sentences), CHUNK)]
order = list(range(len(chunks)))
random.Random(SEED).shuffle(order)
n_test = int(0.10 * len(order)); n_val = int(0.10 * len(order))
test_ids = set(order[:n_test]); val_ids = set(order[n_test:n_test + n_val])

train_s, val_s, test_s = [], [], []
for i, ch in enumerate(chunks):                       # keep original order inside each split
    (test_s if i in test_ids else val_s if i in val_ids else train_s).extend(ch)

for name, part in [("train", train_s), ("val", val_s), ("test", test_s)]:
    print(f"{name:5s}: {len(part):5d} sentences, {sum(len(s.split()) for s in part):6,d} words")

## 4. Baseline: trigram model with backoff

Counts of "two previous words -> next word" learned from the training split. If the trigram context was never seen it backs off to bigram, then to unigram.
This is the classic non-neural baseline every neural model should beat.

In [ ]:
def wtok(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

class BackoffNGram:
    def __init__(self, sentences):
        self.tri = collections.defaultdict(collections.Counter)
        self.bi = collections.defaultdict(collections.Counter)
        self.uni = collections.Counter()
        for s in sentences:
            w = ["<s>", "<s>"] + wtok(s)
            for i in range(2, len(w)):
                self.tri[(w[i-2], w[i-1])][w[i]] += 1
                self.bi[w[i-1]][w[i]] += 1
                self.uni[w[i]] += 1

    def predict(self, context, k=5):
        w = ["<s>", "<s>"] + wtok(context)
        out, seen = [], set()
        for counter in (self.tri.get((w[-2], w[-1])), self.bi.get(w[-1]), self.uni):
            if not counter: continue
            total = sum(counter.values())
            for word, c in counter.most_common(k + len(out)):
                if word not in seen:
                    seen.add(word); out.append((word, c / total))
                if len(out) == k: return out
        return out

ngram = BackoffNGram(train_s)
print(ngram.predict("Sherlock", 5))
print(ngram.predict("I have no doubt that", 5))

## 5. Load the pretrained models (GPT-2 and BERT)

Both are used **as-is** first (zero-shot). Candidate predictions are restricted to real whole words (no punctuation, no word-fragments), and
case variants are merged (`the`/`The`), so all four models are scored the same way.

In [ ]:
gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt0 = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()      # zero-shot GPT-2

bert_tok = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device).eval()

# --- vocabulary masks: only allow whole-word tokens as candidates
gpt_vocab = gpt_tok.get_vocab()                                        # token string -> id
gpt_id2word = {i: t[1:].lower() for t, i in gpt_vocab.items() if re.fullmatch(r"Ġ[A-Za-z]+", t)}
gpt_mask = torch.full((len(gpt_vocab),), float("-inf"))
gpt_mask[list(gpt_id2word)] = 0.0
gpt_mask = gpt_mask.to(device)

bert_vocab = bert_tok.get_vocab()
bert_id2word = {i: t for t, i in bert_vocab.items() if t.isalpha() and not t.startswith("[")}
bert_mask = torch.full((len(bert_vocab),), float("-inf"))
bert_mask[list(bert_id2word)] = 0.0
bert_mask = bert_mask.to(device)

def _top_words(logits, mask, id2word, k):
    probs = torch.softmax(logits + mask, dim=-1)
    top = torch.topk(probs, 50)
    scores = collections.defaultdict(float)
    for p, i in zip(top.values.tolist(), top.indices.tolist()):
        scores[id2word[i]] += p                                        # merge The/the, etc.
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

@torch.no_grad()
def gpt_predict(model, context, k=5):
    ids = gpt_tok(context.rstrip(), return_tensors="pt", truncation=True, max_length=1024).input_ids.to(device)
    logits = model(ids).logits[0, -1]
    return _top_words(logits, gpt_mask, gpt_id2word, k)

@torch.no_grad()
def bert_predict(context, k=5):
    enc = bert_tok(context.rstrip() + " [MASK]", return_tensors="pt", truncation=True, max_length=512).to(device)
    pos = (enc.input_ids[0] == bert_tok.mask_token_id).nonzero()[0, 0]
    logits = bert(**enc).logits[0, pos]
    return _top_words(logits, bert_mask, bert_id2word, k)

print("GPT-2 :", gpt_predict(gpt0, "Mr. Sherlock Holmes was a very"))
print("BERT  :", bert_predict("Mr. Sherlock Holmes was a very"))

## 6. Build the test set (identical for every model)

For each test sentence, every alphabetic word that has at least 3 words of context becomes one example: *context -> next word*.
Only words that are a **single token in both GPT-2 and BERT** are kept, so no model is judged on a word it cannot output in one step.

In [ ]:
def build_examples(sents, min_ctx_words=3):
    ex = []
    for s in sents:
        for m in re.finditer(r"[A-Za-z]+", s):
            st, en = m.span()
            if st == 0 or s[st-1] != " ": continue                    # only words that follow a space
            if en < len(s) and s[en] == "'": continue                  # skip don't / Holmes' style fragments
            ctx = s[:st].rstrip()
            if len(re.findall(r"[A-Za-z]+", ctx)) < min_ctx_words: continue
            gold = m.group()
            if len(gpt_tok.encode(" " + gold)) != 1: continue
            if bert_tok.tokenize(gold.lower()) != [gold.lower()]: continue
            ex.append((ctx, gold.lower()))
    return ex

all_examples = build_examples(test_s)
MAX_EVAL = 2000
test_examples = random.Random(SEED).sample(all_examples, min(MAX_EVAL, len(all_examples)))
print(f"{len(all_examples):,} eligible test examples; evaluating on {len(test_examples):,}")
print("Example:", test_examples[0])

## 7. Zero-shot evaluation (n-gram, BERT, GPT-2)

In [ ]:
def evaluate(predict_fn, examples, ks=(1, 5)):
    hits = {k: 0 for k in ks}
    for ctx, gold in tqdm(examples, leave=False):
        preds = [w for w, _ in predict_fn(ctx, max(ks))]
        for k in ks: hits[k] += gold in preds[:k]
    return {f"top{k}": hits[k] / len(examples) for k in ks}

results = {}
results["Trigram + backoff"] = evaluate(ngram.predict, test_examples)
results["BERT (zero-shot)"] = evaluate(bert_predict, test_examples)
results["GPT-2 (zero-shot)"] = evaluate(lambda c, k: gpt_predict(gpt0, c, k), test_examples)
pd.DataFrame(results).T.style.format("{:.1%}")

## 8. Fine-tune GPT-2 on the book

The training split is tokenized and cut into 128-token blocks; the model learns to predict each next token (standard causal-LM training).
After every epoch we measure **validation perplexity** and keep the best weights, so the model does not over-fit a ~35k-word corpus.

In [ ]:
BLOCK = 128
def make_blocks(sents, tok, block=BLOCK):
    ids = tok(" ".join(sents)).input_ids
    n = len(ids) // block
    return torch.tensor(ids[: n * block]).view(n, block)

train_blocks = make_blocks(train_s, gpt_tok)
val_blocks   = make_blocks(val_s, gpt_tok)
test_blocks  = make_blocks(test_s, gpt_tok)
print("blocks  train/val/test:", len(train_blocks), len(val_blocks), len(test_blocks))

@torch.no_grad()
def perplexity(model, blocks, bs=16):
    model.eval(); total, count = 0.0, 0
    for i in range(0, len(blocks), bs):
        b = blocks[i:i + bs].to(device)
        loss = model(b, labels=b).loss.item()
        n_tok = b.shape[0] * (b.shape[1] - 1)
        total += loss * n_tok; count += n_tok
    return math.exp(total / count)

ppl_zero = {"val": perplexity(gpt0, val_blocks), "test": perplexity(gpt0, test_blocks)}
print("GPT-2 zero-shot perplexity:", {k: round(v, 2) for k, v in ppl_zero.items()})

In [ ]:
EPOCHS, BATCH, LR = 4, 8, 5e-5

ft = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
opt = torch.optim.AdamW(ft.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_blocks) / BATCH)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)

best_val, best_state, history = float("inf"), None, []
for epoch in range(1, EPOCHS + 1):
    ft.train()
    perm = torch.randperm(len(train_blocks))
    run_loss = 0.0
    for i in tqdm(range(0, len(perm), BATCH), desc=f"epoch {epoch}", leave=False):
        b = train_blocks[perm[i:i + BATCH]].to(device)
        loss = ft(b, labels=b).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        run_loss += loss.item()
    val_ppl = perplexity(ft, val_blocks)
    history.append((epoch, run_loss / steps_per_epoch, val_ppl))
    print(f"epoch {epoch}: train loss {run_loss/steps_per_epoch:.3f} | val perplexity {val_ppl:.2f}")
    if val_ppl < best_val:
        best_val = val_ppl
        best_state = {k: v.detach().cpu().clone() for k, v in ft.state_dict().items()}

ft.load_state_dict(best_state); ft.eval()
print("Best validation perplexity:", round(best_val, 2))

## 9. Final results

In [ ]:
results["GPT-2 (fine-tuned)"] = evaluate(lambda c, k: gpt_predict(ft, c, k), test_examples)
table = pd.DataFrame(results).T
table.columns = ["Top-1 accuracy", "Top-5 accuracy"]
display(table.style.format("{:.1%}"))
table.to_csv("results_accuracy.csv")

ppl_ft = perplexity(ft, test_blocks)
print(f"\nGPT-2 test perplexity: zero-shot {ppl_zero['test']:.2f}  ->  fine-tuned {ppl_ft:.2f}")

ax = table.plot(kind="bar", figsize=(8, 4.5), rot=15)
ax.set_ylabel("Accuracy"); ax.set_title("Next-word prediction on held-out text of A Study in Scarlet")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); plt.tight_layout()
plt.savefig("results_accuracy.png", dpi=150); plt.show()

### Side-by-side demo on a few prompts

In [ ]:
prompts = ["Sherlock Holmes took down his", "You have been in Afghanistan, I", "The man was found lying in the", "It is a capital mistake to theorise before"]
for p in prompts:
    print(f"\n>>> {p} ...")
    for name, fn in [("Trigram", ngram.predict), ("BERT", bert_predict),
                     ("GPT-2", lambda c, k: gpt_predict(gpt0, c, k)),
                     ("GPT-2 fine-tuned", lambda c, k: gpt_predict(ft, c, k))]:
        print(f"  {name:17s}", [w for w, _ in fn(p, 5)])

## 10. Error analysis (what to write about in your README)

Look at where the best model fails — it makes the project far more convincing than numbers alone.

In [ ]:
errors = []
for ctx, gold in test_examples[:600]:
    preds = [w for w, _ in gpt_predict(ft, ctx, 5)]
    if gold not in preds: errors.append((ctx[-70:], gold, preds[:3]))
print(f"{len(errors)} of 600 misses in top-5. A random sample:\n")
for c, g, p in random.sample(errors, 10):
    print(f"...{c}  [{g}]  -> predicted {p}")

## 11. Save the model and launch the demo app

Run the first cell to save/download the fine-tuned model; run the second to get a public Gradio link.

In [ ]:
ft.save_pretrained("sherlock-gpt2"); gpt_tok.save_pretrained("sherlock-gpt2")
!zip -qr sherlock-gpt2.zip sherlock-gpt2
from google.colab import files
files.download("sherlock-gpt2.zip")        # or copy the folder to Google Drive instead

In [ ]:
import gradio as gr

MODELS = {
    "GPT-2 fine-tuned on A Study in Scarlet": lambda t, k: gpt_predict(ft, t, k),
    "GPT-2 (zero-shot)": lambda t, k: gpt_predict(gpt0, t, k),
    "BERT (zero-shot)": lambda t, k: bert_predict(t, k),
    "Trigram + backoff": lambda t, k: ngram.predict(t, k),
}

def suggest(text, model_name, k):
    text = (text or "").strip()
    if len(text.split()) < 1:
        return {}
    preds = MODELS[model_name](text, int(k))
    return {w: float(p) for w, p in preds}

demo = gr.Interface(
    fn=suggest,
    inputs=[gr.Textbox(lines=2, label="Start of a sentence", placeholder="Sherlock Holmes took down his"),
            gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Model"),
            gr.Slider(1, 10, value=5, step=1, label="Number of suggestions")],
    outputs=gr.Label(label="Next-word predictions"),
    title="Next-Word Prediction: Sherlock Holmes edition",
    description="Compare a trigram baseline, BERT and GPT-2 (zero-shot and fine-tuned on A Study in Scarlet).",
    examples=[["Sherlock Holmes took down his", list(MODELS)[0], 5],
              ["You have been in Afghanistan, I", list(MODELS)[0], 5],
              ["The man was found lying in the", list(MODELS)[1], 5]],
)
demo.launch(share=True)